In [27]:
import json
from typing import Dict, List, Set

GRAPH_FILE = "data/neutreeko_graph.json"

class Graph:

    def __init__(
        self,
        nodes: List[dict],
        edges: List[dict],
        nodes_map: Dict[str, dict],
        selected_node_id=None,
    ):
        self.nodes: List[dict] = nodes
        self.edges: List[dict] = edges
        self.nodes_map: Dict[str, dict] = nodes_map  # MyNodeData と互換
        self.selected_node_id = selected_node_id

    @classmethod
    def load(cls, path=GRAPH_FILE):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(
            nodes=data.get("nodes", []),
            edges=data.get("edges", []),
            nodes_map=data.get("nodesMap", {}),
            selected_node_id=data.get("selectedNodeId"),
        )

    def save(self, path=GRAPH_FILE):
        data = {
            "nodes":self.nodes,
            "edges": self.edges,
            "nodesMap": self.nodes_map,
            "selectedNodeId": self.selected_node_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    # =====================
    # ノード取得
    # =====================
    def get_node_map(self, node_id: str) -> dict:
        return self.nodes_map.get(node_id)
    

    # =====================
    # ノード一覧取得
    # =====================
    def get_node_maps(self) -> List[dict]:
        return list(self.nodes_map.values())
    
    def get_nodes(self) -> List[dict]:
        return self.nodes

    # =====================
    # 子取得（nodesMap の children を利用）
    # =====================
    def get_children(self, node_id: str) -> List[str]:
        node = self.nodes_map.get(node_id)
        return node.get("children", []) if node else []

    # =====================
    # 子孫取得
    # =====================
    def get_descendants(self, node_id: str) -> Set[str]:
        result = set()
        stack = self.get_children(node_id)
        while stack:
            curr = stack.pop()
            if curr not in result:
                result.add(curr)
                stack.extend(self.get_children(curr))
        return result

    # =====================
    # ノード削除（子は繋ぎ替え or orphan化）
    # =====================
    def delete_node(self, node_id: str):
        if node_id not in self.nodes_map:
            return

        # 親 → children の接続を修正
        for n in self.nodes_map.values():
            if "children" in n:
                n["children"] = [c for c in n["children"] if c != node_id]

        # ノード自体を削除
        del self.nodes_map[node_id]

        # エッジ削除
        self.edges = [
            e for e in self.edges if e["source"] != node_id and e["target"] != node_id
        ]

        # 選択状態の修正
        if self.selected_node_id == node_id:
            self.selected_node_id = None

    # =====================
    # 子孫ごと削除（サブツリー削除）
    # =====================
    def delete_subtree(self, node_id: str):
        targets = self.get_descendants(node_id) | {node_id}
        for tid in list(targets):
            self.delete_node(tid)

    # =====================
    # 色変更
    # =====================
    def set_color(self, node_id: str, color: str):
        if node_id in self.nodes_map:
            self.nodes_map[node_id]["color"] = color

    # =====================
    # エッジ取得
    # =====================
    def get_edges(self):
        return self.edges

    def get_edge(self, source_id: str, target_id: str):
        edges = []
        for e in self.edges:
            if source_id != None and target_id != None:
                if e["source"] == source_id and e["target"] == target_id:
                    edges.append(e)
            elif source_id != None:
                if e["source"] == source_id:
                    edges.append(e)
            elif target_id != None:
                if e["target"] == target_id:
                    edges.append(e)
        return edges
    # =====================
    # 盤面取得
    # =====================
    def get_board(self, node_id: str):
        return self.nodes_map[node_id].get("board")

    # =====================
    # デバッグ
    # =====================
    def print_structure(self):
        print("=== nodesMap structure ===")
        for node_id, data in self.nodes_map.items():
            print(
                f"{node_id}: children={data.get('children')}, color={data.get('color')}"
            )
        print("\n=== edges ===")
        for e in self.edges:
            print(f"{e['source']} -> {e['target']}")

In [39]:
graph = Graph.load()
nodes = graph.get_node_maps() 

for node in nodes:
    # 重複する子ノードを削除
    # children = graph.get_children(node["id"])
    # children = list(set(children))
    # node["children"] = children
    # エッジのないノードを削除
    # from_edges = graph.get_edge(None, node["id"])
    # to_edges = graph.get_edge(node["id"], None)
    # if from_edges == [] and to_edges == []:
    #     graph.delete_node(node["id"])
    #     print(node["id"])
    # parentId を設定
    # from_edges = graph.get_edge(None, node["id"])
    # if len(from_edges) == 1:
    #     parent_id = from_edges[0]["source"]
    #     node["parentId"] = parent_id
    # else:
    #     print(f"Node {node['id']} has {len(from_edges)} parent edges.")
    if "color" not in node:
        print(f"Node {node['id']} has no color. Setting to default (#ffffff).")
        node["color"] = "#ffffff"
        
# for node in nodes:
#     parent = node.get("parentId")
#     if parent:
#         parent_node = graph.get_node_map(parent)
#         if parent_node:
#             if "hiddenChildren" in parent_node and parent_node["hiddenChildren"]:
#                 node["hidden"] = True
#             if "hidden" in parent_node and parent_node["hidden"]:
#                 node["hidden"] = True

In [28]:
node_maps = graph.get_node_maps()
for node in node_maps:
    key = node.keys()
    if "hidden" not in key:
        print(f"Node {node['id']} has no 'hidden' key. Setting to default (False).")
        node["hidden"] = False
    if "hiddenChildren" not in key:
        print(f"Node {node['id']} has no 'hiddenChildren' key. Setting to default (False).")
        node["hiddenChildren"] = False
    if "position" not in key:
        print(f"Node {node['id']} has unexpected keys: {key}")

In [12]:
node_maps = graph.get_node_maps()
all_board_set_b = set()
all_board_set_w = set()
for node in node_maps:
    if node.get("current_player") == "B":
        board_list_b = tuple(tuple((row)) for row in node["board"])
        if board_list_b in all_board_set_b:
            # graph.delete_node(node["id"])
            print(f"Deleted duplicate node {node['id']}")
        else:
            all_board_set_b.add(board_list_b)
    elif node.get("current_player") == "W":
        board_list_w = tuple(tuple((row)) for row in node["board"])
        if board_list_w in all_board_set_w:
            # graph.delete_node(node["id"])
            print(f"Deleted duplicate node {node['id']}")
        else:
            all_board_set_w.add(board_list_w)

In [3]:
graph = Graph.load()
graph.get_edge(None, "node_0")
graph.get_edge("node_0", None)

[{'id': 'e_node_0_node_7',
  'source': 'node_0',
  'target': 'node_7',
  'hidden': False},
 {'id': 'e_node_0_node_8',
  'source': 'node_0',
  'target': 'node_8',
  'hidden': False},
 {'id': 'e_node_0_node_9',
  'source': 'node_0',
  'target': 'node_9',
  'hidden': False},
 {'id': 'e_node_0_node_10',
  'source': 'node_0',
  'target': 'node_10',
  'hidden': False}]

In [26]:
graph.save()

In [23]:
graph = Graph.load()

graph.print_structure()


=== nodesMap structure ===
node_0: children=['node_7', 'node_8', 'node_10', 'node_9'], color=None
node_7: children=['node_46', 'node_42', 'node_45', 'node_48', 'node_51', 'node_52', 'node_39', 'node_40', 'node_41', 'node_53'], color=None
node_8: children=['node_14', 'node_15', 'node_16', 'node_17'], color=None
node_9: children=['node_88', 'node_86', 'node_89', 'node_92', 'node_84', 'node_87', 'node_94', 'node_91', 'node_113', 'node_85', 'node_90', 'node_93', 'node_95', 'node_96', 'node_97'], color=None
node_10: children=['node_79', 'node_76'], color=None
node_14: children=['node_125', 'node_196', 'node_200', 'node_126', 'node_195', 'node_127', 'node_191', 'node_115', 'node_124', 'node_114', 'node_192', 'node_199', 'node_190', 'node_119', 'node_188', 'node_194', 'node_198', 'node_193', 'node_118', 'node_189', 'node_117', 'node_187', 'node_121', 'node_197'], color=#ffffff
node_15: children=[], color=None
node_16: children=[], color=None
node_17: children=[], color=None
node_39: children=